### Deep Learning (U-Net 3D); Notebook 06

#### Objectif
Segmenter le thrombus voxel par voxel avec un **U-Net 3D**, puis l'évaluer avec **la même FROC patient que O4**
(sensibilité à 1/2/4/8 FP/patient sur les 33 porteurs) pour comparer honnêtement classique vs deep learning.

#### Prérequis
- **Environnement** : WSL2 `EP-Ubuntu`, venv `~/venvs/ep` (TensorFlow 2.21 + GPU).

 ⚠️ Le kernel Windows `EP_env` n'a **pas** de GPU.

- **Données** : `data/processed/PATxxx/volume.npy` + `mask.npy` (O1, orientation GT corrigée)
- **Matériel** : RTX 4060 8 Go → patchs à résolution native, batch 1–2

#### Étapes principales
1. Vérification GPU
2. Test rapide du U-Net 3D (instanciation + mémoire GPU)
3. Dataset de patchs *(à venir)*
4. Entraînement *(à venir)*
5. Évaluation FROC + Dice + volume *(à venir)*

> **Pourquoi des patchs ?** Réduire le volume entier à 128×128×64 (ancienne approche) divise la résolution par 4 :
> un caillot périphérique de 2–3 mm disparaît. Or 67 % des emboles FUMPE sont périphériques.


### 1. Vérification GPU

Doit afficher `GPU:0`. Si la liste est vide : mauvais kernel (pas le venv `ep` de WSL) ou `LD_LIBRARY_PATH` cusolver absent.
`set_memory_growth` évite que TensorFlow réserve les 8 Go d'un coup.


In [ ]:
# Cellule 1 - Imports + GPU
import os, sys
import numpy as np
import tensorflow as tf

root_path = os.path.abspath(os.path.join('..'))
if root_path not in sys.path: sys.path.insert(0, root_path)

gpus = tf.config.list_physical_devices('GPU')
for g in gpus:
    tf.config.experimental.set_memory_growth(g, True)
print(f'TensorFlow {tf.__version__} | GPU : {gpus}')
assert gpus, 'Aucun GPU : vérifier le kernel (venv ep dans WSL)'

### 2. Test rapide du U-Net 3D

On instancie le `UNet3D` existant (`src/O5_deep_learning/unet_3d.py`) en sortie binaire (`num_classes=1`, sigmoïde)
sur un patch de la taille prévue pour l'entraînement (64×96×96), puis on mesure le pic de mémoire GPU
d'un pas avant + arrière. Objectif : vérifier que l'architecture **s'instancie sous Keras 3** et **tient dans 8 Go**.

> **Sortie attendue :** forme de sortie `(2, 64, 96, 96, 1)`, nombre de paramètres, pic mémoire GPU (< ~6 Go souhaité).


In [ ]:
# Cellule 2 - Test rapide U-Net 3D (instanciation + pic mémoire d'un pas d'entraînement)
from src.O5_deep_learning.unet_3d import UNet3D

PATCH = (64, 96, 96)   # (Z, Y, X) à résolution native
BATCH = 2

model = UNet3D(input_shape=(*PATCH, 1), num_classes=1, features_start=16, depth=4)
x = tf.random.normal((BATCH, *PATCH, 1))
y = tf.cast(tf.random.uniform((BATCH, *PATCH, 1)) > 0.99, tf.float32)

tf.config.experimental.reset_memory_stats('GPU:0')
with tf.GradientTape() as tape:
    out = model(x, training=True)
    loss = tf.reduce_mean(tf.keras.losses.binary_crossentropy(y, out))
grads = tape.gradient(loss, model.trainable_variables)

peak = tf.config.experimental.get_memory_info('GPU:0')['peak'] / 1024**3
print(f'Sortie       : {out.shape}')
print(f'Paramètres   : {model.count_params():,}')
print(f'Gradients    : {sum(g is not None for g in grads)}/{len(grads)} non nuls')
print(f'Pic mém. GPU : {peak:.2f} Go (batch={BATCH}, float32)')